In [1]:
# User Access Anomaly Detection Project
# Dataset: CSV file (user_access_logs.csv)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

df = pd.read_csv('user_access_logs.csv')
print("Dataset Shape:", df.shape)
print(df.head())

categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# Fill missing values if any
df.fillna(0, inplace=True)

# Separate features and target
if 'CLASS_LABEL' in df.columns:  # Supervised
    X = df.drop(['CLASS_LABEL'], axis=1)
    y = df['CLASS_LABEL']
else:  # Unsupervised
    X = df
    y = None

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    # Evaluation
    print("✅ Accuracy:", accuracy_score(y_test, y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

#Unsupervised Approach (if labels are not available)

if y is None:
    iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    iso_forest.fit(X_scaled)
    y_pred_unsupervised = iso_forest.predict(X_scaled)
    # Convert predictions: -1 (anomaly) -> 1, 1 (normal) -> 0
    y_pred_unsupervised = np.where(y_pred_unsupervised == -1, 1, 0)
    df['Anomaly_Prediction'] = y_pred_unsupervised
    print("\nUnsupervised Anomaly Detection Results:")
    print(df['Anomaly_Prediction'].value_counts())



Dataset Shape: (1000, 10)
   UserID            LoginTime           LogoutTime   IP_Address DeviceType  \
0    1051  2025-01-01 00:00:00  2025-01-01 01:00:00     10.0.0.1    Desktop   
1    1092  2025-01-01 01:00:00  2025-01-01 02:00:00   172.16.0.1     Mobile   
2    1014  2025-01-01 02:00:00  2025-01-01 03:00:00  192.168.1.1     Mobile   
3    1071  2025-01-01 03:00:00  2025-01-01 04:00:00   172.16.0.1     Mobile   
4    1060  2025-01-01 04:00:00  2025-01-01 05:00:00     10.0.0.1     Mobile   

  Location  FailedLoginAttempts  SessionDuration ResourceAccessed  CLASS_LABEL  
0      USA                    2               47        Dashboard            0  
1  Germany                    1               70            Admin            0  
2      USA                    2               78            Admin            0  
3      USA                    3               65        Dashboard            0  
4  Germany                    0               47        Dashboard            0  
✅ Accuracy: 0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
